**Modelo clasificador de imagenes generadas con DeepFake vs Reales**

*Dataset obtenido de:* https://www.kaggle.com/datasets/muhammadbilal6305/200k-real-vs-ai-visuals-by-mbilal/data

In [ ]:
import kagglehub

path = kagglehub.dataset_download("muhammadbilal6305/200k-real-vs-ai-visuals-by-mbilal")

print("Path to dataset files:", path)

**Preprocesamiento**

In [ ]:
import os
import numpy as np
import pandas as pd


In [ ]:
images_dir_path = path + "/my_real_vs_ai_dataset/my_real_vs_ai_dataset/"

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 123
VAL_SPLIT = 0.2

class_names = ['ai_images', 'real']
train_paths, train_labels = [], []
val_paths, val_labels = [], []

rng = np.random.default_rng(SEED)

for idx, class_name in enumerate(class_names):
    class_dir = os.path.join(images_dir_path, class_name)
    files = np.array([os.path.join(class_dir, f) for f in os.listdir(class_dir)])
    rng.shuffle(files)  # shuffle SOLO dentro de esta clase

    n_val = int(len(files) * VAL_SPLIT)
    val_paths.extend(files[:n_val])
    val_labels.extend([idx] * n_val)
    train_paths.extend(files[n_val:])
    train_labels.extend([idx] * (len(files) - n_val))

print("Train:", np.bincount(train_labels))
print("Val:  ", np.bincount(val_labels))

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label

def make_dataset(paths, labels, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    return ds

train_ds = make_dataset(train_paths, train_labels, shuffle=True)
val_ds = make_dataset(val_paths, val_labels, shuffle=False)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

**Entrenamiento**

In [ ]:
from tensorflow import keras
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

IMG_SIZE = (224, 224)
EPOCHS_TRAIN_1 = 10
EPOCHS_TRAIN_2 = 100
PATIENCE = 3

data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.1)
])

base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(*IMG_SIZE, 3)
)

base_model.trainable = False

model = keras.Sequential([
    data_augmentation,
    keras.layers.Lambda(preprocess_input),
    base_model,

    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.4),

    keras.layers.Dense(2, activation='softmax')
])

model.compile(
    optimizer= Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience= PATIENCE,
    restore_best_weights=True
)

model_checkpoint = ModelCheckpoint(
    './content/best_model.keras',
    monitor='val_accuracy',
    save_best_only=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=PATIENCE,
    min_lr=1e-7
)

history1 = model.fit(
    train_ds,
    epochs=EPOCHS_TRAIN_1,
    validation_data=val_ds,
    callbacks=[
        early_stopping,
        reduce_lr,
        model_checkpoint
    ],
    verbose=1
)

model.compile(
    optimizer= Adam(learning_rate= 1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
  )

base_model.trainable = True

for layer in base_model.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

for layer in base_model.layers[:-50]:
    layer.trainable = False

history2 = model.fit(
    train_ds,
    epochs=EPOCHS_TRAIN_2,
    validation_data=val_ds,
    callbacks=[
        early_stopping,
        model_checkpoint,
        reduce_lr
        ],
    verbose=1
)

In [ ]:
model.save("./content/DvR_model_v2.keras")

In [ ]:
y_pred_probs = model.predict(val_ds)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true = np.concatenate([y for x, y in val_ds], axis=0)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_true, y_pred_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels= class_names)
disp.plot(cmap="Blues")


In [ ]:
import matplotlib.pyplot as plt

idx = 0

for images, labels in val_ds:
    batch_size = images.shape[0]

    for i in range(batch_size):
        real = y_true[idx]
        pred = y_pred_classes[idx]

        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(f"Real: {real} | Predicho: {pred}")
        plt.axis("off")
        plt.show()

        idx += 1